<a href="https://colab.research.google.com/github/jadenvv/selfAttentionSentiment/blob/main/selfAttentionSentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
from google.colab import userdata
os.environ['KAGGLE_KEY']= userdata.get("Kaggle_key")
os.environ['KAGGLE_USERNAME']= userdata.get("Kaggle_user")
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset

In [101]:

training_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,"jp797498e/twitter-entity-sentiment-analysis", "twitter_training.csv")
training_df.columns = ["ID","TOPIC",'SENTIMENT', "TWEET"]
training_df.drop_duplicates(subset="TWEET", inplace=True,  ignore_index=True)
training_df.dropna(subset=["TWEET", "SENTIMENT"], inplace=True, ignore_index=True)
sent,reverse_indices  = np.unique(training_df["SENTIMENT"], return_inverse=True)
training_df.SENTIMENT = reverse_indices
training_df['TWEET'] = [x+ " <EOS>"for x in training_df["TWEET"]]
vocab = list(set((" ".join(training_df["TWEET"].tolist())).split(' ')))
stoi = {text:idx for idx,text in enumerate(vocab)}
itos = {idx:text for idx,text in enumerate(vocab)}


Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.


In [102]:

class SentDataset(Dataset):
  def __init__(self,df):
      self.tweet_data= [
          np.array([stoi[x] for x in tweet.split(' ')])
          for tweet in df.TWEET
      ]
      self.sentiment = df["SENTIMENT"]


  def __getitem__(self,idx):
      tweet = self.tweet_data.iloc[idx]
      sentiment = self.sentiment.iloc[idx]
      return (tweet, sentiment)
  def __len__(self):
      return len(self.tweet_data)

In [103]:
dataset_sent  = SentDataset(training_df)